# Neural A\* — FINN Synthesis

**Run on the compute server in a CPU Slurm session, inside tmux.** The final build takes
over an hour and dies with the SSH connection if tmux isn't holding it.

```bash
tmux new -s finn
srun --time=06:00:00 --mem=32G --pty bash
conda activate fpga_hw2
source /opt/Xilinx/Vivado/2022.1/settings64.sh
source /opt/Xilinx/Vitis_HLS/2022.1/settings64.sh
jupyter lab --no-browser --ip=0.0.0.0 --port=XXXX
```

Needs `neural_astar_encoder_w4a8.onnx` from `02_quantize.ipynb`.

### The question

`01_baseline` established that the CNN is 92% of runtime, so it's the thing worth
accelerating. `02_quantize` established that W4A8 reproduces float quality at 190 KB.
This notebook answers the one question left: **can a Zynq-7020 run this network
faster than the CPU already does?**

The arithmetic says it will be close and might not. 399 MMAC per frame, and the
fourth convolution alone is 76% of that. To hit 10 ms at 100 MHz roughly
400 MACs every cycle, built out of LUTs because 4-bit weights don't use DSPs.
A Z-7020 has 53,200 LUTs total.

A negative answer here is still a result — arguably a better one than a marginal
positive, because it comes with a frontier showing exactly where the crossover
sits. What matters is that the number is measured rather than asserted.

### Route

1. Inspect the ONNX before handing it to FINN.
2. **Estimate-only build** — minutes, no Vivado. Gives real layer names and
   per-layer cycle counts.
3. Set folding from those numbers.
4. Full build with bitfile — an hour plus.
5. Compare against the CPU baseline honestly.

## 0. Environment

In [1]:
import os, sys, json, subprocess, shutil, time
import numpy as np

WORKDIR = os.path.expanduser("~/fpga_hw2")
os.chdir(WORKDIR)

# Vivado needs libtinfo.so.5 from the conda env
conda_lib = os.path.join(sys.prefix, "lib")
os.environ["LD_LIBRARY_PATH"] = conda_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["FINN_ROOT"] = os.path.expanduser("~/finn")
os.environ["XILINX_VIVADO"] = "/opt/Xilinx/Vivado/2022.1"
os.environ["XILINX_HLS"] = "/opt/Xilinx/Vitis_HLS/2022.1"

ONNX = os.path.join(WORKDIR, "neural_astar_encoder_w4a8.onnx")
assert os.path.exists(ONNX), f"missing {ONNX} -- run 02_quantize.ipynb first"
print(f"model: {os.path.basename(ONNX)}  ({os.path.getsize(ONNX)//1024} KB)")

r = subprocess.run(["vivado", "-version"], capture_output=True, text=True)
assert r.returncode == 0, "Vivado not on PATH -- source the settings64.sh files"
print("vivado:", (r.stdout + r.stderr).strip().splitlines()[0])

import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
print("FINN imported OK")

PYNQ_Z2 = {"LUT": 53200, "FF": 106400, "BRAM_36K": 140, "DSP": 220}
CLK_NS = 10.0                      # 100 MHz

model: neural_astar_encoder_w4a8.onnx  (1541 KB)
vivado: Vivado v2022.1 (64-bit)
FINN imported OK


## 1. Read the graph before FINN does

Worth two minutes. FINN's failure messages point at whichever transformation
tripped, not at the model construct that caused it, so knowing what's in the
graph first saves a lot of guessing later.

In [2]:
import onnx
from collections import Counter

model = onnx.load(ONNX)
ops = Counter(n.op_type for n in model.graph.node)
print("nodes:", dict(ops))

inp = model.graph.input[0]
out = model.graph.output[0]
print("input :", [d.dim_value for d in inp.type.tensor_type.shape.dim])
print("output:", [d.dim_value for d in out.type.tensor_type.shape.dim])

if ops.get("Sigmoid"):
    print("\n*** Sigmoid present -- see section 6 before starting the full build. ***")

nodes: {'Quant': 11, 'Conv': 5, 'BatchNormalization': 5, 'Relu': 4, 'Sigmoid': 1}
input : [1, 2, 32, 32]
output: [1, 1, 32, 32]

*** Sigmoid present -- see section 6 before starting the full build. ***


### The Sigmoid

FINN's streamlining has no rule for `Sigmoid`. It may survive `step_streamline`
and then fail at `step_convert_to_hw` with no hardware layer to map it to.

The reasoning that put it there is still sound: sigmoid is monotonic, so
`Sigmoid → Quant` is mathematically a `MultiThreshold` with thresholds at
`logit(level_boundary)`. Whether *this* FINN version performs that folding is a
question only the build answers. Section 6 has the fallback, which needs about
three minutes of GPU time and no new code from us.

Try the estimate build as-is first. If step 6 or 7 fails, jump to section 6.

## 2. Estimate-only build

FINN's full flow ends in Vivado place-and-route, which is where the hour goes.
But `ESTIMATE_REPORTS` alone stops before any HLS or synthesis and takes
minutes — and it gives us the two things needed to configure folding
intelligently: the **real layer names** after graph conversion, and the
**per-layer cycle counts**.

Guessing folding parameters and finding out an hour later is the mistake this
step exists to avoid.

In [3]:
# generate_outputs controls what FINN WRITES, not which steps it RUNS. Without
# an explicit `steps` list it executes all 19 -- including HLS synthesis and
# Vivado -- so an "estimate-only" build would still take an hour and would trip
# HLS constraints that estimates do not care about.
def _estimate_steps():
    steps = getattr(build_cfg, "estimate_only_dataflow_steps", None)
    if steps:
        return steps
    full = build_cfg.default_build_dataflow_steps
    stop = full.index("step_generate_estimate_reports") + 1
    return full[:stop]


ESTIMATE_STEPS = _estimate_steps()
print(f"estimate-only flow: {len(ESTIMATE_STEPS)} steps "
      f"(full flow is {len(build_cfg.default_build_dataflow_steps)})")


def make_cfg(onnx_path, output_dir, build_dir, folding=None,
             specialize=None, bitfile=False, clk_ns=CLK_NS):
    os.environ["FINN_BUILD_DIR"] = build_dir
    os.makedirs(build_dir, exist_ok=True)

    outputs = [build_cfg.DataflowOutputType.ESTIMATE_REPORTS]
    kwargs = {}
    if bitfile:
        outputs += [
            build_cfg.DataflowOutputType.BITFILE,
            build_cfg.DataflowOutputType.PYNQ_DRIVER,
            build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE,
        ]
    else:
        kwargs["steps"] = ESTIMATE_STEPS      # stop before HLS

    return build_cfg.DataflowBuildConfig(
        output_dir=output_dir,
        board="Pynq-Z2",
        shell_flow_type=build_cfg.ShellFlowType.VIVADO_ZYNQ,
        synth_clk_period_ns=clk_ns,
        generate_outputs=outputs,
        folding_config_file=folding,
        specialize_layers_config_file=specialize,
        auto_fifo_depths=False,
        verify_save_rtlsim_waveforms=False,
        **kwargs,
    )


def fresh(*dirs):
    for d in dirs:
        p = os.path.join(WORKDIR, d)
        if os.path.exists(p):
            shutil.rmtree(p, ignore_errors=True)
            print(f"removed stale {d}")


SPECIALIZE = os.path.join(WORKDIR, "specialize_astar.json")
with open(SPECIALIZE, "w") as f:
    json.dump({"Defaults": {"preferred_impl_style": ["rtl", "all"]}}, f, indent=2)

EST_OUT = os.path.join(WORKDIR, "finn_est")
fresh("finn_est", "finn_est_tmp")

print("estimate-only build -- expect a few minutes\n")
t0 = time.time()
build.build_dataflow_cfg(ONNX, make_cfg(
    ONNX, EST_OUT, os.path.join(WORKDIR, "finn_est_tmp"),
    specialize=SPECIALIZE, bitfile=False))
print(f"\ndone in {time.time()-t0:.0f}s")

estimate-only flow: 10 steps (full flow is 19)
removed stale finn_est
removed stale finn_est_tmp
estimate-only build -- expect a few minutes

Building dataflow accelerator from $HOME/project/neural_astar_encoder_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/finn_est_tmp
Final outputs will be generated in $HOME/project/finn_est
Build log is at $HOME/project/finn_est/build_dataflow.log
Running step: step_qonnx_to_finn [1/10]
Running step: step_tidy_up [2/10]
Running step: step_streamline [3/10]
Running step: step_convert_to_hw [4/10]
Running step: step_create_dataflow_partition [5/10]
Running step: step_specialize_layers [6/10]
Running step: step_target_fps_parallelization [7/10]
Running step: step_apply_folding_config [8/10]
Running step: step_minimize_bit_width [9/10]
Running step: step_generate_estimate_reports [10/10]
Completed successfully

done in 3s


## 3. What the estimates say

`estimate_layer_cycles.json` is the important one. In a FINN dataflow every layer
runs concurrently, so **throughput is set by the slowest layer**, not the sum.
One badly folded layer caps the whole accelerator no matter how well the others
are configured.

In [4]:
def load_report(output_dir, name):
    p = os.path.join(output_dir, "report", name)
    if not os.path.exists(p):
        print(f"missing report: {name}")
        return None
    with open(p) as f:
        return json.load(f)


cycles = load_report(EST_OUT, "estimate_layer_cycles.json")
res_est = load_report(EST_OUT, "estimate_layer_resources.json")

if cycles:
    worst = max(cycles.values())
    total_lut = sum(v.get("LUT", 0) for k, v in (res_est or {}).items()
                    if k != "total")
    print(f"{'layer':<38} {'cycles':>12} {'share':>7}")
    print("-" * 60)
    for k, v in sorted(cycles.items(), key=server kv: -kv[1]):
        print(f"{k:<38} {v:12,d} {100*v/worst:6.0f}%")
    print("-" * 60)
    print(f"{'BOTTLENECK':<38} {worst:12,d}")
    print(f"\nthroughput  : {worst * CLK_NS / 1e6:.2f} ms/frame at "
          f"{1000/CLK_NS:.0f} MHz")
    print(f"              {1e9/(worst*CLK_NS):.1f} frames/s")
    print("\nUnfolded, every layer runs at SIMD=PE=1 -- one MAC per cycle.")
    print("Section 4 fixes that.")

layer                                        cycles   share
------------------------------------------------------------
                                          2,359,296    100%
------------------------------------------------------------
BOTTLENECK                                2,359,296

throughput  : 23.59 ms/frame at 100 MHz
              42.4 frames/s

Unfolded, every layer runs at SIMD=PE=1 -- one MAC per cycle.
Section 4 fixes that.


In [5]:
if res_est:
    print(f"{'layer':<38} {'LUT':>8} {'BRAM':>7} {'DSP':>5}")
    print("-" * 62)
    for k, v in res_est.items():
        if k == "total":
            continue
        print(f"{k:<38} {v.get('LUT',0):8.0f} {v.get('BRAM_18K',0):7.0f} "
              f"{v.get('DSP',0):5.0f}")
    tot = res_est.get("total", {})
    print("-" * 62)
    print(f"{'TOTAL':<38} {tot.get('LUT',0):8.0f} {tot.get('BRAM_18K',0):7.0f} "
          f"{tot.get('DSP',0):5.0f}")
    print(f"{'PYNQ-Z2 available':<38} {PYNQ_Z2['LUT']:8d} "
          f"{PYNQ_Z2['BRAM_36K']*2:7d} {PYNQ_Z2['DSP']:5d}")
    print("\nThese are FINN's estimates, not synthesis results -- they run "
          "optimistic.\nSection 7 has the real numbers.")

layer                                       LUT    BRAM   DSP
--------------------------------------------------------------
                                              0       1     1
--------------------------------------------------------------
TOTAL                                         0       1     1
PYNQ-Z2 available                         53200     280   220

These are FINN's estimates, not synthesis results -- they run optimistic.
Section 7 has the real numbers.


## 4. Folding

`SIMD` is input-side parallelism, `PE` output-side; a layer computes `SIMD × PE`
MACs per cycle and takes `(MW/SIMD) × (MH/PE) × pixels` cycles.

Three constraints, and the third only bites at step 11:

* `SIMD` must divide `MW`, `PE` must divide `MH`.
* `SIMD` must also divide the layer's input channels, for the
  `ConvolutionInputGenerator` feeding it.
* **`SIMD ≥ MW/1024`** — an HLS `MatrixVectorActivation` limit. An unfolded
  layer at `SIMD=1` with `MW=1152` fails this, which is why the estimate build
  must stop before HLS rather than merely skip its outputs.

Where the work sits, computed from the architecture:

| layer | in→out | MW | MH | MACs | share |
|---|---|---|---|---|---|
| conv0 | 2→32 | 18 | 32 | 0.6 M | 0.1% |
| conv1 | 32→64 | 288 | 64 | 18.9 M | 4.7% |
| conv2 | 64→128 | 576 | 128 | 75.5 M | 18.9% |
| conv3 | 128→256 | 1152 | 256 | **302 M** | **75.6%** |
| conv4 | 256→1 | 2304 | 1 | 2.4 M | 0.6% |

Three quarters of the work is in one layer, so that layer gets the parallelism.
Allocating `SIMD × PE` in proportion to each layer's MAC count balances the
pipeline — no point making conv1 fast while conv3 stalls everything:

| layer | SIMD | PE | MAC/cycle | cycles |
|---|---|---|---|---|
| conv0 | 2 | 1 | 2 | 294,912 |
| conv1 | 4 | 4 | 16 | 1,179,648 |
| conv2 | 8 | 8 | 64 | 1,179,648 |
| conv3 | 16 | 16 | 256 | 1,179,648 |
| conv4 | 4 | 1 | 4 | 589,824 |

Three layers land on exactly the same cycle count, which is what a balanced
pipeline looks like. **342 MACs/cycle → 11.8 ms/frame at 100 MHz.**

Against the CPU's 6.7 ms, that is *slower*. Getting to parity needs roughly
double the parallelism, and whether 684 LUT-based MACs fit alongside everything
else is exactly what section 7 measures.

The cell below maps this onto whatever names FINN actually generated, rather
than assuming them.

In [6]:
# MACs/cycle per conv, in graph order -- scale together to trade area for speed.
TARGET_FOLDING = [(2, 1), (4, 4), (8, 8), (16, 16), (4, 1)]

# (MW, MH) per conv, in graph order -- used to match folding to real layers.
CONV_DIMS = [(18, 32), (288, 64), (576, 128), (1152, 256), (2304, 1)]

import glob
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp


def discover_layers(output_dir):
    # Read the real node names and dimensions out of FINN's intermediate model.
    # Names cannot be predicted: step_specialize_layers gives each layer an rtl
    # or hls implementation and NUMBERS THE TWO FAMILIES SEPARATELY, so the result is
    # e.g. MVAU_rtl_0, MVAU_hls_0, MVAU_rtl_1. Sorting those by trailing integer
    # yields duplicate indices and the wrong order.
    # model.graph.node is topologically ordered, so that order is authoritative.
    pats = ["*specialize_layers*.onnx", "*apply_folding*.onnx", "*.onnx"]
    path = None
    for pat in pats:
        hits = sorted(glob.glob(os.path.join(output_dir,
                                             "intermediate_models", pat)))
        if hits:
            path = hits[0] if pat != "*.onnx" else hits[-1]
            break
    if path is None:
        raise RuntimeError(f"no intermediate models under {output_dir}")

    model = ModelWrapper(path)
    layers = []
    for node in model.graph.node:
        if "MVAU" not in node.op_type and "VVAU" not in node.op_type:
            continue
        try:
            inst = getCustomOp(node)
            mw, mh = inst.get_nodeattr("MW"), inst.get_nodeattr("MH")
        except Exception:
            mw = mh = None
        layers.append((node.name, node.op_type, mw, mh))
    return os.path.basename(path), layers


def build_folding_config(output_dir, target, path, dims=CONV_DIMS):
    src, layers = discover_layers(output_dir)
    print(f"layer names read from {src}")
    if not layers:
        raise RuntimeError("no MVAU/VVAU nodes found in the graph")

    by_dim = {(mw, mh): fold for (mw, mh), fold in zip(dims, target)}
    cfg = {"Defaults": {}}
    fallback = []

    print()
    print(f"{'node':<24} {'op_type':<14} {'MW':>6} {'MH':>5} {'SIMD':>5} {'PE':>4}")
    print("-" * 64)
    for i, (name, op, mw, mh) in enumerate(layers):
        fold = by_dim.get((mw, mh))
        if fold is None and i < len(target):
            fold = target[i]                      # fall back to graph order
            fallback.append(name)
        if fold is None:
            print(f"{name:<24} {op:<14} {str(mw):>6} {str(mh):>5}   SKIPPED")
            continue
        simd, pe = fold
        cfg[name] = {"SIMD": simd, "PE": pe}
        print(f"{name:<24} {op:<14} {str(mw):>6} {str(mh):>5} {simd:5d} {pe:4d}")

    if fallback:
        print()
        print(f"WARNING: matched by graph order, not dimensions: {fallback}")

    missing = [n for n, _, _, _ in layers if n not in cfg]
    if missing:
        raise RuntimeError(f"no folding assigned to {missing} -- these would run "
                           f"at SIMD=1 and fail HLS codegen at step 11")

    with open(path, "w") as f:
        json.dump(cfg, f, indent=2)
    print()
    print(f"wrote {os.path.basename(path)} ({len(cfg)-1} layers configured)")
    return cfg



def check_folding(target, dims=CONV_DIMS):
    # Verify the constraints HLS enforces, before an hour of building:
    #   SIMD divides MW, PE divides MH, and SIMD >= MW/1024.
    # The last one is an HLS MatrixVectorActivation limit that an unfolded
    # SIMD=1 layer violates, and it only surfaces at step 11.
    ok = True
    print(f"{'layer':7} {'MW':>6} {'MH':>5} {'SIMD':>5} {'PE':>4} "
          f"{'MW/SIMD':>8} {'MAC/cyc':>8} {'cycles':>11}")
    for i, ((mw, mh), (simd, pe)) in enumerate(zip(dims, target)):
        problems = []
        if mw % simd:
            problems.append(f"SIMD={simd} does not divide MW={mw}")
        if mh % pe:
            problems.append(f"PE={pe} does not divide MH={mh}")
        if simd < mw / 1024:
            problems.append(f"SIMD={simd} < MW/1024={mw/1024:.2f}")
        cyc = (mw // simd) * (mh // pe) * 1024
        print(f"conv{i}   {mw:6d} {mh:5d} {simd:5d} {pe:4d} {mw//simd:8d} "
              f"{simd*pe:8d} {cyc:11,d}")
        for p in problems:
            print(f"        !! {p}")
            ok = False
    worst = max((mw // s) * (mh // p) * 1024
                for (mw, mh), (s, p) in zip(dims, target))
    print()
    print(f"bottleneck {worst:,} cycles = {worst*CLK_NS/1e6:.2f} ms "
          f"at {1000/CLK_NS:.0f} MHz")
    print("VALID" if ok else "INVALID -- fix before building")
    return ok


assert check_folding(TARGET_FOLDING), "folding config violates HLS constraints"

FOLDING = os.path.join(WORKDIR, "folding_astar.json")
build_folding_config(EST_OUT, TARGET_FOLDING, FOLDING)

layer       MW    MH  SIMD   PE  MW/SIMD  MAC/cyc      cycles
conv0       18    32     2    1        9        2     294,912
conv1      288    64     4    4       72       16   1,179,648
conv2      576   128     8    8       72       64   1,179,648
conv3     1152   256    16   16       72      256   1,179,648
conv4     2304     1     4    1      576        4     589,824

bottleneck 1,179,648 cycles = 11.80 ms at 100 MHz
VALID
layer names read from step_specialize_layers.onnx

node                     op_type            MW    MH  SIMD   PE
----------------------------------------------------------------
                         MVAU_hls           18    32     2    1
                         MVAU_hls          288    64     4    4
                         MVAU_hls          576   128     8    8
                         MVAU_hls         1152   256    16   16
                         MVAU_rtl         2304     1     4    1

wrote folding_astar.json (1 layers configured)


{'Defaults': {}, '': {'SIMD': 4, 'PE': 1}}

In [7]:
# Re-estimate with folding applied. Still minutes, still no Vivado.
EST2_OUT = os.path.join(WORKDIR, "finn_est_folded")
fresh("finn_est_folded", "finn_est_folded_tmp")

t0 = time.time()
build.build_dataflow_cfg(ONNX, make_cfg(
    ONNX, EST2_OUT, os.path.join(WORKDIR, "finn_est_folded_tmp"),
    folding=FOLDING, specialize=SPECIALIZE, bitfile=False))
print(f"done in {time.time()-t0:.0f}s\n")

cyc2 = load_report(EST2_OUT, "estimate_layer_cycles.json")
res2 = load_report(EST2_OUT, "estimate_layer_resources.json")

if cyc2:
    worst2 = max(cyc2.values())
    print(f"{'layer':<38} {'cycles':>12}")
    for k, v in sorted(cyc2.items(), key=server kv: -kv[1]):
        print(f"{k:<38} {v:12,d}")
    print(f"\nthroughput : {worst2*CLK_NS/1e6:.2f} ms/frame")
    print(f"CPU baseline (01_baseline, batch=1): 6.70 ms/frame")

if res2:
    t = res2.get("total", {})
    print(f"\n{'':10} {'LUT':>9} {'BRAM_18K':>10} {'DSP':>6}")
    print(f"{'estimated':10} {t.get('LUT',0):9.0f} {t.get('BRAM_18K',0):10.0f} "
          f"{t.get('DSP',0):6.0f}")
    print(f"{'available':10} {PYNQ_Z2['LUT']:9d} {PYNQ_Z2['BRAM_36K']*2:10d} "
          f"{PYNQ_Z2['DSP']:6d}")
    over = [k for k, a in [("LUT", PYNQ_Z2["LUT"]),
                           ("BRAM_18K", PYNQ_Z2["BRAM_36K"]*2),
                           ("DSP", PYNQ_Z2["DSP"])] if t.get(k, 0) > a]
    if over:
        print(f"\nOVER BUDGET on {over} -- halve the folding and re-estimate")
        print("before starting the full build.")
    else:
        print("\nFits (by estimate). Estimates run optimistic; synthesis decides.")

removed stale finn_est_folded
removed stale finn_est_folded_tmp
Building dataflow accelerator from $HOME/project/neural_astar_encoder_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/finn_est_folded_tmp
Final outputs will be generated in $HOME/project/finn_est_folded
Build log is at $HOME/project/finn_est_folded/build_dataflow.log
Running step: step_qonnx_to_finn [1/10]
Running step: step_tidy_up [2/10]
Running step: step_streamline [3/10]
Running step: step_convert_to_hw [4/10]
Running step: step_create_dataflow_partition [5/10]
Running step: step_specialize_layers [6/10]
Running step: step_target_fps_parallelization [7/10]
Running step: step_apply_folding_config [8/10]
Running step: step_minimize_bit_width [9/10]
Running step: step_generate_estimate_reports [10/10]
Completed successfully
done in 3s

layer                                        cycles
MVAU_hls_3                              301,989,888
MVAU_hls_2                               75,497,472
MVAU_hls_1    

Iterate here, not in section 5. Each estimate pass costs minutes; each full build
costs an hour. Scale `TARGET_FOLDING` up until it exceeds the budget, then step back
one notch.

Halving every entry gives ~171 MACs/cycle and ~24 ms; doubling gives ~684 and
~6 ms, if it fits.

## 5. Full build

Now Vivado runs. **Over an hour**, and it must be inside tmux.

Bitfile generation is worth doing even if the throughput number disappoints,
because post-synthesis resource utilisation is a headline result in its own
right: it's what says whether the search engine could ever share this die with
the CNN.

In [ ]:
BUILD_OUT = os.path.join(WORKDIR, "finn_output_astar")
fresh("finn_output_astar", "finn_build_astar_tmp")

print("Full build with bitfile -- over an hour. Keep tmux attached.\n")
t0 = time.time()
build.build_dataflow_cfg(ONNX, make_cfg(
    ONNX, BUILD_OUT, os.path.join(WORKDIR, "finn_build_astar_tmp"),
    folding=FOLDING, specialize=SPECIALIZE, bitfile=True))
print(f"\nbuild complete in {(time.time()-t0)/60:.0f} minutes")

removed stale finn_output_astar
removed stale finn_build_astar_tmp
Full build with bitfile -- over an hour. Keep tmux attached.

Building dataflow accelerator from $HOME/project/neural_astar_encoder_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/finn_build_astar_tmp
Final outputs will be generated in $HOME/project/finn_output_astar
Build log is at $HOME/project/finn_output_astar/build_dataflow.log
Running step: step_qonnx_to_finn [1/19]
Running step: step_tidy_up [2/19]
Running step: step_streamline [3/19]
Running step: step_convert_to_hw [4/19]
Running step: step_create_dataflow_partition [5/19]
Running step: step_specialize_layers [6/19]
Running step: step_target_fps_parallelization [7/19]
Running step: step_apply_folding_config [8/19]
Running step: step_minimize_bit_width [9/19]
Running step: step_generate_estimate_reports [10/19]
Running step: step_hw_codegen [11/19]
> $HOME/finn/src/finn/custom_op/fpgadataflow/hls/matrixvectoractivation_hls.py(201)defines()
    

Traceback (most recent call last):
  File "$HOME/finn/src/finn/builder/build_dataflow.py", line 158, in build_dataflow_cfg
    model = transform_step(model, cfg)
  File "$HOME/finn/src/finn/builder/build_dataflow_steps.py", line 515, in step_hw_codegen
    model = model.transform(PrepareIP(cfg._resolve_fpga_part(), cfg._resolve_hls_clk_period()))
  File "$HOME/miniforge3/envs/fpga_hw2/lib/python3.10/site-packages/qonnx/core/modelwrapper.py", line 145, in transform
    (transformed_model, model_was_changed) = transformation.apply(transformed_model)
  File "$HOME/finn/src/finn/transformation/fpgadataflow/prepare_ip.py", line 94, in apply
    _codegen_single_node(node, model, self.fpgapart, self.clk)
  File "$HOME/finn/src/finn/transformation/fpgadataflow/prepare_ip.py", line 54, in _codegen_single_node
    inst.code_generation_ipgen(model, fpgapart, clk)
  File "$HOME/finn/src/finn/custom_op/fpgadataflow/hlsbackend.py", line 120, in code_generation_ipgen
    self.defines("ipgen")
  File 

## 6. If FINN rejected the Sigmoid

Skip this unless the build failed at `step_streamline` or `step_convert_to_hw`
with a Sigmoid-related error.

**The fix:** drop the sigmoid from the hardware and apply it on the host. The
accelerator emits the pre-sigmoid value; the host computes `sigmoid()` over 1024
numbers, which against a CNN that dominates runtime is free. This is exactly how
FINN handles classifiers — the accelerator returns logits and the host does the
softmax.

The output quantizer has to be recalibrated, because pre-sigmoid activations
have a different range than the (0,1) sigmoid output. A short distillation
against pre-sigmoid targets handles it. Run this in `02_quantize.ipynb`, where
the training data is already loaded, then re-export and come back.

In [ ]:
SIGMOID_FALLBACK = '''
# --- paste into 02_quantize.ipynb, after the sweep ---
import torch, torch.nn as nn, brevitas.nn as qnn
from brevitas.quant import Int8ActPerTensorFloat

class QuantEncoderNoSigmoid(QuantGuidanceEncoder):
    # Same network, sigmoid moved to the host: forward() returns the
    # pre-sigmoid value. Apply torch.sigmoid() to whatever comes back from
    # the accelerator to recover the guidance map.
    def forward(self, x):
        return self.out_q(self.model(self.input_q(x)))

base = sweep_results[CHOSEN]["model"]
model_ns = QuantEncoderNoSigmoid(*CHOSEN)
model_ns.load_state_dict(base.state_dict(), strict=False)
model_ns.to(DEVICE).train()

# Targets are now pre-sigmoid, so recover them from the float model.
float_encoder.eval()
with torch.no_grad():
    Y_pre = torch.cat([float_encoder.model(X_train[i:i+256].to(DEVICE)).cpu()
                       for i in range(0, len(X_train), 256)], 0)

opt = torch.optim.Adam(model_ns.parameters(), lr=1e-4)
for ep in range(10):
    perm = torch.randperm(len(X_train)); tot = 0.0
    for i in range(0, len(X_train), BATCH):
        idx = perm[i:i+BATCH]
        opt.zero_grad()
        loss = nn.L1Loss()(model_ns(X_train[idx].to(DEVICE)),
                           Y_pre[idx].to(DEVICE))
        loss.backward(); opt.step(); tot += loss.item()
    print(f"  epoch {ep+1}  L1 {tot/(len(X_train)//BATCH):.5f}")

# Wrap with sigmoid so the A* metrics stay comparable.
class HostSigmoid(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return torch.sigmoid(self.m(x))

print(eval_encoder(HostSigmoid(model_ns.eval())))
export_and_validate(model_ns.eval(),
                    "neural_astar_encoder_w4a8_nosigmoid.onnx")
'''
print(SIGMOID_FALLBACK)

## 7. Post-synthesis resources

The real numbers. FINN's estimates are consistently optimistic — they ignore
routing, control logic and FIFO overhead — so this is the table that goes in the
report.

In [ ]:
def print_resources(output_dir, label=""):
    res = load_report(output_dir, "post_synth_resources.json")
    if res is None:
        print("no post-synthesis report -- build incomplete?")
        return None
    top = res.get("(top)", {})
    print(f"{label}\n{'Resource':<12} {'Used':>9} {'Available':>11} {'Util':>8}")
    print("-" * 44)
    for key, avail in PYNQ_Z2.items():
        used = top.get(key, top.get(key.replace("_36K", ""), 0))
        print(f"{key:<12} {used:9.0f} {avail:11d} {100*used/avail:7.1f}%")
    return top


final = print_resources(BUILD_OUT, "Neural A* encoder, W4A8")

timing = load_report(BUILD_OUT, "post_route_timing.json") or {}
if timing:
    print("\ntiming:", timing)

## 8. The verdict

The comparison the whole project has been building toward. State it plainly —
including if it's unfavourable.

In [ ]:
CPU_MS = 6.70          # from 01_baseline, batch=1, 32x32
SEARCH_MS = 0.60       # A* search on CPU, unchanged by any of this

cyc_final = load_report(BUILD_OUT, "estimate_layer_cycles.json")
if cyc_final:
    fpga_ms = max(cyc_final.values()) * CLK_NS / 1e6
    print(f"{'':28} {'CNN':>9} {'search':>9} {'total':>9}")
    print("-" * 58)
    print(f"{'CPU (float32)':28} {CPU_MS:8.2f}ms {SEARCH_MS:8.2f}ms "
          f"{CPU_MS+SEARCH_MS:8.2f}ms")
    print(f"{'PYNQ-Z2 W4A8 + CPU search':28} {fpga_ms:8.2f}ms {SEARCH_MS:8.2f}ms "
          f"{fpga_ms+SEARCH_MS:8.2f}ms")
    speedup = (CPU_MS + SEARCH_MS) / (fpga_ms + SEARCH_MS)
    print(f"\nend-to-end speedup: {speedup:.2f}x")
    print()
    if speedup < 1.0:
        print("Slower than the CPU. That is a result, not a failure -- report the")
        print("frontier and the device class where it inverts. Amdahl said the CNN")
        print("was the only stage worth accelerating; this says a Z-7020 lacks the")
        print("compute to accelerate it. Both are measurements.")
    else:
        print("Faster than the CPU. Report the resource cost alongside, and note")
        print("that batch=1 is the honest comparison for a planner.")

    needed = max(cyc_final.values()) / (CPU_MS * 1e6 / CLK_NS)
    print(f"\nParallelism multiple needed for CPU parity: {needed:.1f}x current")

## 9. What to write down

1. **Post-synthesis utilisation** at the chosen folding — LUT, FF, BRAM, DSP.
   The headline hardware result.
2. **Achieved `Fmax`** versus the 100 MHz target. If timing didn't close,
   every throughput number above is optimistic and must be restated.
3. **Throughput versus CPU**, with the folding that produced it, and the
   resource cost of the next step up.
4. **Where the frontier crosses.** Given measured LUTs per MAC/cycle, how much
   fabric would parity need? That converts "the Z-7020 is too small" into a
   quantitative statement about the device class this workload actually wants.
5. **The Sigmoid**, if it caused trouble — whether FINN folded it or it was moved
   to the host, and what that cost.

Then `04_board.ipynb`: load the bitfile, run `finn_test_vectors.npz` through it,
confirm the hardware reproduces the model, and measure real wall-clock instead of
estimated cycles.